In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from concept_abstraction.training import train_model, train_ppo_model
from concept_abstraction.selection import greedy_selection, random_selection, human_centered_selection, greedy_selection_diverse, human_centered_selection_diverse
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
from concept_abstraction.environments import *
from sklearn.metrics import accuracy_score
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
import gymnasium as gym
from collections import Counter


ImportError: cannot import name 'greedy_selection_diverse' from 'concept_abstraction.selection' (/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/selection.py)

In [ ]:
is_jupyter = 'ipykernel' in sys.modules

In [ ]:
if is_jupyter: 
    seed        = 42
    environment_string = "cartpole"
    concept_retrieval = "raw"
    show_baseline = True 
    human_accuracy_by_concept = None 
    target_abstraction = 0.05
    out_folder = "cartpole"
    num_concepts_selected = 4
    cbm_accuracy_by_concept = None 
    human_reliance_by_concept = None 
    reward_error = 0.1
    transition_error = 0
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
    parser.add_argument('--environment_nodes', help='Size of the environment; number of nodes', type=int, default=4)
    parser.add_argument('--show-baseline', action='store_true', help='Whether to show the baseline')
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--human_accuracy_by_concept', nargs='*', type=float, default=None)
    parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
    parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
    parser.add_argument('--human_reliance_by_concept', help="How much does AI rely on human intervention?",  nargs='*', type=float, default=None)
    parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
    parser.add_argument('--transition_error', help="How much to perturb the transition by?", type=float, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string
    environment_nodes = args.environment_nodes 
    show_baseline = args.show_baseline
    num_concepts_selected = args.num_concepts_selected
    human_accuracy_by_concept = args.human_accuracy_by_concept
    human_reliance_by_concept = args.human_reliance_by_concept
    target_abstraction = args.target_abstraction
    cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
    reward_error = args.reward_error
    transition_error = args.transition_error
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [ ]:
results = {}
results['parameters'] = {'seed'      : seed,
        'environment_string'    : environment_string, 
        'show_baseline': show_baseline,
        'num_concepts_selected': num_concepts_selected,
        'human_accuracy_by_concept': human_accuracy_by_concept, 
        'human_reliance_by_concept': human_reliance_by_concept, 
        'target_abstraction': target_abstraction,
        'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
        'reward_error': reward_error, 
        'transition_error': transition_error,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cartpole', 'show_baseline': True, 'num_concepts_selected': 4, 'human_accuracy_by_concept': None, 'human_reliance_by_concept': None, 'target_abstraction': 0.05, 'cbm_accuracy_by_concept': None, 'reward_error': 0.1, 'transition_error': 0}


In [ ]:
np.random.seed(seed)
random.seed(seed)

In [ ]:
def make_env_fn(concept_list,accuracies=None,binary=False,aggregated=False,llm=False,reward_error=0):
    env = gym.make("CartPole-v1")

    if binary:
        env = DiscretizeObservationWrapper(env, bins_per_feature=4)
        env = BinaryObservationSubsetWrapper(env, concept_list,accuracies)
        env.concepts = list(range(16))
    elif aggregated:
        env = get_binary_subset_env(golden_model, env, concept_list,accuracies=accuracies)
    elif llm:
        env = CustomBinaryFeatureWrapper(env)
        env = BinaryObservationSubsetWrapper(env, concept_list,accuracies=accuracies)
    else:
        env = ObservationSubsetWrapper(env, indices=concept_list)

    if reward_error > 0:
        env = RewardPerturbationWrapper(env,reward_error)

    return env

In [ ]:
total_timesteps = 20000

## Retrieving Concepts

In [22]:
env = make_env_fn([0,1,2,3],None,reward_error=1)
model = train_ppo_model(env,total_timesteps=total_timesteps)
env = make_env_fn([0,1,2,3],None)
get_average_reward(env,model)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


0.3122

In [30]:
env = make_env_fn([0,1,2,3],None)
model = train_ppo_model(env,total_timesteps=total_timesteps)
env = make_env_fn([0,1,2,3],None)
get_average_reward(env,model)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


0.3161

In [85]:
env = make_env_fn([0,1],None)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


31.0

In [87]:
env = make_env_fn([2,3],None)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

248.0

In [97]:
env = make_env_fn([0],None,binary=True)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

40.0

In [22]:
env = make_env_fn(list(range(16)),None,binary=True)
model = train_ppo_model(env,total_timesteps=total_timesteps)


In [29]:
d = get_observed_transition(model,env)

In [32]:
for key in d:
    for s in d[key]:
        print(d[key][s])

4
21
23
1
11
5
1
1
22
10
22
3
53
8
1
1
126
10
57
32
2
6
2
109
7
62
15
33
9
45
4
3
6
6
6
61
12
8
23
8
61
27
4
2
4
1
2
1
2
3
35
9
44
72
10
17
15
2
19
4
1
6
3
3
38
57
28
8
7
48
44
19
3
2
3
7
10
4
1
7
5
1
1
2
1
6
14
162
57
36
2
8
4
2
2
5
3
3
2
2
12
22
4
14
37
4
1
1
8
7
1
171
31
64
1
10
7
1
3
4
16
11
3
1
2
4
15
1
5
5
11
19
7
1
2
2
1
2
18
1
7
1
1
1
6
1
6
1
1
1
1


In [20]:
env.observation_space

MultiBinary(16)

In [96]:
env = make_env_fn(list(range(16)),None,binary=True)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


251.0

In [10]:
env = make_env_fn([0,1,2,3],None)
golden_model = train_ppo_model(env,total_timesteps=100000)
get_average_reward(golden_model,env)
# TODO: Save the golden model (dependent on the reward error)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


0.5

In [11]:
transitions,rewards = get_observed_transition(golden_model,env)
all_states = list(transitions.keys())
all_states_array = np.array([[float(j) for j in i.split(" ")] for i in all_states])
all_binarized_concepts = [state_to_concepts(i,binary=True) for i in all_states_array]

In [57]:
# Level 1: Policy
actions = golden_model.predict(all_states_array)[0].reshape(-1,1)

In [15]:
# Level 2: Q Function
state_dim = env.observation_space.shape[0]
action_dim = 1
q_estimator = SimpleQEstimator(state_dim, action_dim, golden_model)

# 3. Train (this does everything)
q_estimator.collect_and_train(env, num_episodes=100)

# 4. Use Q-function
q_estimator

Episode 0/100
Episode 10/100
Episode 20/100
Episode 30/100
Episode 40/100
Episode 50/100
Episode 60/100
Episode 70/100
Episode 80/100
Episode 90/100
Training Q-network...


/usr0/home/naveenr/.local/lib/python3.8/site-packages/torch/nn/modules/loss.py:538: UserWarning: Using a target size (torch.Size([256, 1])) that is different to the input size (torch.Size([256])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr0/home/naveenr/.local/lib/python3.8/site-packages/torch/nn/modules/loss.py:538: UserWarning: Using a target size (torch.Size([80, 1])) that is different to the input size (torch.Size([80])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0, Average Loss: 4528.4605
Epoch 10, Average Loss: 607.6860
Epoch 20, Average Loss: 606.5520
Epoch 30, Average Loss: 606.2449
Epoch 40, Average Loss: 606.0661


In [92]:
q_values = np.array([[q_estimator.get_q_value(all_states_array[i],[0]),q_estimator.get_q_value(all_states_array[i],[1])] for i in range(len(all_states_array))])

In [48]:
# Level 3: Reward + Q Function
reward_list = [rewards[all_states[i]] for i in range(len(all_states)) if all_states[i] in rewards]

transitions_by_concept = {}
for idx,state in enumerate(all_states):
    concept = tuple(all_binarized_concepts[idx])
    if concept not in transitions_by_concept:
        transitions_by_concept[concept] = {'0': [], '1': []}
    for (action,next_state) in transitions[state]:
        next_concept = state_to_concepts(next_state,binary=True)
        transitions_by_concept[concept][action].append(tuple(next_concept))
for key in transitions_by_concept:
    for action in ['0','1']:
        transitions_by_concept[key][action] = Counter(transitions_by_concept[key][action])
all_concepts = set(transitions_by_concept.keys())

for key in transitions_by_concept:
    for action in ['0','1']:
        for val in transitions_by_concept[key][action]:
            if val not in all_concepts:
                all_concepts.add(val)
concept_to_idx = {}
for idx,concept in enumerate(all_concepts):
    concept_to_idx[concept] = idx
transitions_vector_by_concept = {}
for key in transitions_by_concept:
    vec = np.zeros(len(concept_to_idx)*2)

    for action in ['0','1']:
        for key_2 in transitions_by_concept[key][action]:
            vec[int(action)*len(concept_to_idx) + concept_to_idx[key_2]] += transitions_by_concept[key][action][key_2]
    vec[:len(concept_to_idx)] = vec[:len(concept_to_idx)]/np.sum(vec[:len(concept_to_idx)]+0.00001)
    vec[len(concept_to_idx):] = vec[len(concept_to_idx):]/np.sum(vec[len(concept_to_idx):]+0.00001)
    transitions_vector_by_concept[key] = vec
list_of_transitions = np.array([transitions_vector_by_concept[tuple(k)] for k in all_binarized_concepts])

In [27]:
num_concepts_selected=2
all_concepts = list(range(16))

In [28]:
selected_concepts = []
random_times = [] 
average_rewards = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > len(concepts):
        break 
    random_concepts = random_selection(env,k)
    env = make_env_fn(random_concepts,accuracies=None,binary=True,reward_error=reward_error)
    model = train_ppo_model(env,total_timesteps=total_timesteps)
    selected_concepts.append(random_concepts)
    env = make_env_fn(random_concepts,accuracies=None,binary=True)
    average_rewards.append(get_average_reward(env,model))
    random_times.append(time.time()-start)

results['random_selection'] = {
    'concepts': [i.tolist() for i in selected_concepts], 
    'values': average_rewards,
    'time': random_times, 
}

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [61]:
greedy_concepts = greedy_selection_diverse(env,2,all_binarized_concepts,actions)

[13, 0]

In [98]:
selected_concepts = []
greedy_average_rewards = []
greedy_times = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > len(env.concepts):
        break 
    
    greedy_concepts = greedy_selection_diverse(env,k,all_binarized_concepts,list_of_transitions)
    env = make_env_fn(greedy_concepts,accuracies=None,binary=True,reward_error=reward_error)
    model = train_ppo_model(env,total_timesteps=total_timesteps)
    selected_concepts.append(greedy_concepts)
    env = make_env_fn(greedy_concepts,accuracies=None,binary=True)
    greedy_average_rewards.append(get_average_reward(env,model))
    greedy_times.append(time.time()-start)

results['greedy_selection'] = {
    'concepts': selected_concepts, 
    'values': greedy_average_rewards,
    'time': greedy_times,
}

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [84]:
all_binarized_concepts[1897],all_binarized_concepts[4228]

(array([0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0], dtype=int8),
 array([0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0], dtype=int8))

In [101]:
if True:# human_accuracy_by_concept is not None or cbm_accuracy_by_concept is not None:
    if human_accuracy_by_concept is None:
        modified_acc_rate = cbm_accuracy_by_concept
    elif cbm_accuracy_by_concept is None:
        modified_acc_rate = human_accuracy_by_concept
    else:
        modified_acc_rate = [reliance_percent*human_acc + (1-reliance_percent)*machine_acc 
                for human_acc,machine_acc,reliance_percent in zip(human_accuracy_by_concept,
                                                                cbm_accuracy_by_concept,
                                                                human_reliance_by_concept)]

    selected_concepts = human_centered_selection_diverse(env,np.random.random(16),0.05,np.array(all_binarized_concepts),q_values)
    selected_concepts = [idx for idx,i in enumerate(selected_concepts) if i>=0.5]

    env = make_env_fn(selected_concepts,accuracies=modified_acc_rate,binary=True)
    model = train_model(env,total_timesteps=total_timesteps)
    env = make_env_fn(selected_concepts,accuracies=None,binary=True)
    human_perf = get_average_reward(env,model)

    results['uncertainty'] = {
        'selected_concepts': selected_concepts,
        'combined_accuracies': modified_acc_rate,
        'combined_value': human_perf,
    }

Set parameter DualReductions to value 0
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (linux64)

CPU model: Intel(R) Core(TM) i7-7700K CPU @ 4.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 49 rows, 17 columns and 224 nonzeros
Model fingerprint: 0xb1fab9af
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-01, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 20 rows, 17 columns, 63 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.9260309e-01   1.000000e+00   0.000000e+00      0s
       2    9.8520026e-01   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  9.852002564e-01


In [102]:
results['uncertainty']

{'selected_concepts': [10, 13],
 'combined_accuracies': None,
 'combined_value': 0.0606}

In [184]:
env = make_env_fn([0,1,2,3],None,aggregated=True)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

Generating training data for threshold learning...
State ranges:
  cart_pos: [-0.179, 0.111]
  cart_vel: [-0.432, 0.424]
  pole_angle: [-0.047, 0.042]
  pole_vel: [-0.539, 0.516]

cart_pos thresholds:
  20th percentile: -0.099
  40th percentile: -0.061
  60th percentile: -0.030
  80th percentile: 0.002

cart_vel thresholds:
  20th percentile: -0.165
  40th percentile: -0.038
  60th percentile: 0.018
  80th percentile: 0.147

pole_angle thresholds:
  20th percentile: -0.004
  40th percentile: -0.001
  60th percentile: 0.002
  80th percentile: 0.004

pole_vel thresholds:
  20th percentile: -0.232
  40th percentile: -0.037
  60th percentile: 0.027
  80th percentile: 0.235


10.0

In [3]:
env = make_env_fn([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],None,aggregated=True)


NameError: name 'make_env_fn' is not defined

In [183]:
env = make_env_fn([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],None,aggregated=True)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

Generating training data for threshold learning...
State ranges:
  cart_pos: [-0.154, 0.094]
  cart_vel: [-0.395, 0.400]
  pole_angle: [-0.044, 0.047]
  pole_vel: [-0.549, 0.516]

cart_pos thresholds:
  20th percentile: -0.064
  40th percentile: -0.025
  60th percentile: -0.010
  80th percentile: 0.013

cart_vel thresholds:
  20th percentile: -0.167
  40th percentile: -0.027
  60th percentile: 0.015
  80th percentile: 0.055

pole_angle thresholds:
  20th percentile: -0.004
  40th percentile: -0.001
  60th percentile: 0.001
  80th percentile: 0.004

pole_vel thresholds:
  20th percentile: -0.170
  40th percentile: -0.040
  60th percentile: 0.009
  80th percentile: 0.239


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


102.0

In [187]:
env = make_env_fn([0,1,2,3,4,5,6,7,8,9,10,11,12],None,llm=True)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


453.0

In [188]:
env = make_env_fn([0,1,2],None,llm=True)
model = train_ppo_model(env,total_timesteps=total_timesteps)
get_average_reward(env,model)

74.0

## Retrieving Concept Values

In [70]:
if environment_string == 'tree':
    total_timesteps = 40000
else:
    total_timesteps = 10000

In [72]:
baseline_concepts = get_baseline_concept_sets(environment_string,environment_nodes)
results['baseline'] = {'concepts': baseline_concepts}
env = make_env_fn(baseline_concepts[-1],None)

In [73]:
if show_baseline:
    values_by_concept = []
    rewards = []
    transitions = []

    for concept_list in baseline_concepts:
        env = make_env_fn(concept_list,None,reward_error,transition_error)
        model = train_model(env,total_timesteps=total_timesteps)
        env = make_env_fn(concept_list,None)
        values_by_concept.append(get_values(env, model))
        rewards.append(env.rewards.tolist())
        transitions.append(env.transitions.tolist())

    results['baseline'] = {
        'concepts': baseline_concepts,
        'values': values_by_concept,
        'rewards': rewards, 
        'transitions': transitions 
    }

## Concept Selection

In [77]:
selected_concepts = []
random_times = [] 
values_by_random_concept = []
rewards = []
transitions = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > len(env.concepts):
        break 
    random_concepts = random_selection(env,k)

    env = make_env_fn(random_concepts,None,reward_error=reward_error,transition_error=transition_error)
    model = train_model(env,total_timesteps=total_timesteps)
    selected_concepts.append(random_concepts)
    env = make_env_fn(random_concepts,None)
    values_by_random_concept.append(get_values(env,model))
    random_times.append(time.time()-start)
    rewards.append(env.rewards.tolist())
    transitions.append(env.transitions.tolist())

results['random_selection'] = {
    'concepts': [i.tolist() for i in selected_concepts], 
    'values': values_by_random_concept,
    'time': random_times, 
    'rewards': rewards, 
    'transitions': transitions 
}

In [78]:
selected_concepts = []
values_by_greedy_concept = []
greedy_times = []
rewards = []
transitions = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > len(env.concepts):
        break 
    greedy_concepts = greedy_selection(env,k)
    env = make_env_fn(greedy_concepts,None,reward_error=reward_error,transition_error=transition_error)
    model = train_model(env,total_timesteps=total_timesteps)
    selected_concepts.append(greedy_concepts)
    env = make_env_fn(greedy_concepts,None)
    values_by_greedy_concept.append(get_values(env,model))
    greedy_times.append(time.time()-start)
    rewards.append(env.rewards.tolist())
    transitions.append(env.transitions.tolist())

results['greedy_selection'] = {
    'concepts': selected_concepts, 
    'values': values_by_greedy_concept,
    'time': greedy_times,
    'rewards': rewards, 
    'transitions': transitions 
}

## Performance under Uncertainty

In [79]:
if human_accuracy_by_concept is not None or cbm_accuracy_by_concept is not None:
    if human_accuracy_by_concept is None:
        modified_acc_rate = cbm_accuracy_by_concept
    elif cbm_accuracy_by_concept is None:
        modified_acc_rate = human_accuracy_by_concept
    else:
        modified_acc_rate = [reliance_percent*human_acc + (1-reliance_percent)*machine_acc 
                for human_acc,machine_acc,reliance_percent in zip(human_accuracy_by_concept,
                                                                cbm_accuracy_by_concept,
                                                                human_reliance_by_concept)]

    selected_concepts = human_centered_selection(env,modified_acc_rate,target_abstraction)
    selected_concepts = [idx for idx,i in enumerate(selected_concepts) if i>=0.5]

    env = make_env_fn(selected_concepts,modified_acc_rate)
    model = train_model(env,total_timesteps=total_timesteps)
    env = make_env_fn(selected_concepts,None)
    human_perf = get_values(env,model)

    concepts = []
    values_error = []

    for idx,concept_list in enumerate(baseline_concepts):
        env = make_env_fn(concept_list,modified_acc_rate)
        model = train_model(env,total_timesteps=total_timesteps)
        concepts.append(concept_list)
        env = make_env_fn(concept_list,None)
        values_error.append(get_values(env,model))

    results['uncertainty'] = {
        'values': values_error,
        'concepts': concepts, 
        'selected_concepts': selected_concepts,
        'combined_accuracies': modified_acc_rate,
        'combined_value': human_perf,
    }

## Save Data

In [80]:
save_path = get_save_path(out_folder,save_name)

In [81]:
delete_duplicate_results(out_folder,"",results)

In [83]:
json.dump(results,open('../../results/'+save_path,'w'))